# Silver — Journeys

Cleans `journeys.csv` (bronze) into a validated, enriched parquet file.

**Transformations applied:**
- Drop corrupt journeys: `total_seconds < 600` (< 10 min) or `> 2700` (> 45 min)
- Drop rows with missing stop timestamps
- Add segment times in minutes between consecutive stops
- Add time features: `hour`, `day_of_week`, `date`, `is_weekend`

Output: `datastreaming/data/silver_journeys.parquet`

In [ ]:
import os
import pandas as pd
from pathlib import Path

DATA_DIR   = Path(os.getenv("DATA_DIR", "../../data"))
SILVER_DIR = Path("../data")
SILVER_DIR.mkdir(exist_ok=True)

MIN_S, MAX_S = 600, 2700  # 10 min – 45 min plausible corridor range

pd.set_option("display.float_format", "{:.1f}".format)

## Load bronze

In [ ]:
df = pd.read_csv(DATA_DIR / "journeys.csv")
print(f"Bronze rows: {len(df):,}")
df["total_seconds"] = pd.to_numeric(df["total_seconds"], errors="coerce")

## Filter corrupt journeys

In [ ]:
too_short = df["total_seconds"] < MIN_S
too_long  = df["total_seconds"] > MAX_S
null_ts   = df["total_seconds"].isna()

print(f"Dropped (< {MIN_S//60}m):  {too_short.sum():,}")
print(f"Dropped (> {MAX_S//60}m):  {too_long.sum():,}")
print(f"Dropped (null):         {null_ts.sum():,}")

df = df[~too_short & ~too_long & ~null_ts].copy()
print(f"\nSilver rows: {len(df):,}")

## Parse timestamps

In [ ]:
AT_COLS = [
    "sincai_at", "marasesti_at", "sf_gheorghe_at",
    "universitate_at", "nicolae_balcescu_at", "arthur_verona_at", "romana_at"
]
for col in AT_COLS:
    df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

# Drop rows where any stop timestamp is missing
missing = df[AT_COLS].isna().any(axis=1)
print(f"Dropped (missing stop timestamp): {missing.sum():,}")
df = df[~missing].copy()

## Add segment times (minutes between consecutive stops)

In [ ]:
SEGMENTS = [
    ("sincai_to_marasesti",          "sincai_at",           "marasesti_at"),
    ("marasesti_to_sf_gheorghe",      "marasesti_at",        "sf_gheorghe_at"),
    ("sf_gheorghe_to_universitate",   "sf_gheorghe_at",      "universitate_at"),
    ("universitate_to_balcescu",      "universitate_at",     "nicolae_balcescu_at"),
    ("balcescu_to_verona",            "nicolae_balcescu_at", "arthur_verona_at"),
    ("verona_to_romana",              "arthur_verona_at",    "romana_at"),
]
for name, start, end in SEGMENTS:
    df[name + "_min"] = (df[end] - df[start]).dt.total_seconds() / 60

df["total_minutes"] = df["total_seconds"] / 60

seg_cols = [s[0] + "_min" for s in SEGMENTS]
print(df[seg_cols + ["total_minutes"]].describe().round(1))

## Add time features

In [ ]:
local = df["sincai_at"].dt.tz_convert("Europe/Bucharest")
df["hour"]        = local.dt.hour
df["day_of_week"] = local.dt.day_name()
df["date"]        = local.dt.date
df["is_weekend"]  = local.dt.dayofweek >= 5

print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"Hours with data: {sorted(df['hour'].unique())}")

## Save silver parquet

In [ ]:
out = SILVER_DIR / "silver_journeys.parquet"
df.to_parquet(out, index=False)
print(f"Saved {len(df):,} rows → {out}")
df.head(3)